# Invariants de similitude

In [86]:
from sage.all import *

## 1. Forme de Smith sur $k[T]$ 

In [ ]:
#Entrée: f, g ∈ k[T]
#Sortie: (d, u, v, a, b) où d = pgcd(f,g) est choisi normalisé unitaire,
#        u*f + v*g = d (Bézout), a = f/d, b = g/d (cofacteurs).

def _bezout_polynomes(f, g):
    
    R = f.parent()
    zero, un = R(0), R(1)

    
    if f == 0 and g == 0:
        return zero, zero, zero, zero, zero
    if f == 0:
        cd = g.leading_coefficient()
        return g / cd, zero, un / cd, zero, un
    if g == 0:
        cd = f.leading_coefficient()
        return f / cd, un / cd, zero, un, zero

    r_prec, r_cour = f, g
    s_prec, s_cour = un, zero
    t_prec, t_cour = zero, un

    
    
    while r_cour != 0:
        q, r_suiv = r_prec.quo_rem(r_cour)
        r_prec, r_cour = r_cour, r_suiv
        s_prec, s_cour = s_cour, s_prec - q * s_cour
        t_prec, t_cour = t_cour, t_prec - q * t_cour

    cd = r_prec.leading_coefficient()
    d = r_prec / cd
    u = s_prec / cd
    v = t_prec / cd

    assert u * f + v * g == d, '_bezout_polynomes: relation de Bézout violée'
    a = f // d
    b = g // d
    assert a * d == f and b * d == g, '_bezout_polynomes: divisions a,b inexactes'
    return d, u, v, a, b



#Remplace (lig_k, lig_i) par (u x lig_k + v x lig_i, -b x lig_k + a x lig_i)
def _op_ligne(P, k, i, u, v, a, b):
    nb_col = P.ncols()
    lig_k = [P[k, j] for j in range(nb_col)]
    lig_i = [P[i, j] for j in range(nb_col)]
    for j in range(nb_col):
        P[k, j] =  u*lig_k[j] + v*lig_i[j]
        P[i, j] = -b*lig_k[j] + a*lig_i[j]


# analogue sur les colonnes

def _op_colonne(P, k, j, u, v, a, b):
    nb_lig = P.nrows()
    col_k = [P[i, k] for i in range(nb_lig)]
    col_j = [P[i, j] for i in range(nb_lig)]
    for i in range(nb_lig):
        P[i, k] =  u*col_k[i] + v*col_j[i]
        P[i, j] = -b*col_k[i] + a*col_j[i]

### Algo de réduction

ON applique l'algorithme de Smith

In [ ]:
#Entrée: A \in M_{m,n}(k[T])
#Sortie: (D, U, V) avec U*A*V = D forme de Smith,
#     U \in GL_m(k[T]), V \in GL_n(k[T]),
#     D diagonale avec d_1 | d_2 | … | d_r unitaires.

def forme_smith(A):
    
    R = A.base_ring()
    m, n = A.nrows(), A.ncols()

    M = matrix(R, m, n, list(A.list()))
    U = identity_matrix(R, m)
    V = identity_matrix(R, n)

    for k in range(min(m, n)):

        while True:
            # Pivot de degré minimal dans le sous-bloc actif
            meilleur_degre = Infinity
            pi, pj = -1, -1
            for i in range(k, m):
                for j in range(k, n):
                    e = M[i, j]
                    if e != 0:
                        d = e.degree()
                        if d < meilleur_degre:
                            meilleur_degre, pi, pj = d, i, j

            if pi == -1:
                return M, U, V

            if pi != k:
                M.swap_rows(k, pi);  U.swap_rows(k, pi)
            if pj != k:
                M.swap_columns(k, pj);  V.swap_columns(k, pj)

            
            
            # Élimination de la k-ième ligne et colonne par Bézout
            modifie = True
            while modifie:
                modifie = False

                for i in range(k + 1, m):
                    if M[i, k] == 0:
                        continue
                    d, u, v, a, b = _bezout_polynomes(M[k, k], M[i, k])
                    _op_ligne(M, k, i, u, v, a, b)
                    _op_ligne(U, k, i, u, v, a, b)
                    modifie = True

                for j in range(k + 1, n):
                    if M[k, j] == 0:
                        continue
                    d, u, v, a, b = _bezout_polynomes(M[k, k], M[k, j])
                    _op_colonne(M, k, j, u, v, a, b)
                    _op_colonne(V, k, j, u, v, a, b)
                    modifie = True

            
            #Vérification de la condition de divisibilité
            mauvais_i, mauvais_j = -1, -1
            for i in range(k + 1, m):
                for j in range(k + 1, n):
                    if M[i, j] != 0:
                        _, r = M[i, j].quo_rem(M[k, k])
                        if r != 0:
                            mauvais_i, mauvais_j = i, j
                            break
                if mauvais_i != -1:
                    break

            if mauvais_i == -1:
                #Normalisation: coefficient dominant = 1
                cd = M[k, k].leading_coefficient()
                M.rescale_row(k, ~cd)
                U.rescale_row(k, ~cd)
                break

            #Restauration de la divisibilité : absorbe la ligne mauvais_i dans la ligne pivot
            for j in range(n):
                M[k, j] += M[mauvais_i, j]
            for j in range(m):
                U[k, j] += U[mauvais_i, j]

    return M, U, V

### Vérification et tests

`_verifier_smith(A, D, U, V, label)` contrôle : $UAV = D$, diagonalité de $D$, condition $d_k \mid d_{k+1}$. Les diagonales sont comparées à l'implémentation native de Sage sur six cas-tests.

In [89]:
def _verifier_smith(A, D, U, V, label=''):
    valide = True

    if U * A * V != D:
        print(f'[ECHEC] {label} : U*A*V ≠ D')
        valide = False

    for i in range(D.nrows()):
        for j in range(D.ncols()):
            if i != j and D[i, j] != 0:
                print(f'[ECHEC] {label}: D non diagonale en ({i},{j})')
                valide = False

    diag = [D[i, i] for i in range(min(D.nrows(), D.ncols())) if D[i, i] != 0]
    for idx in range(1, len(diag)):
        _, r = diag[idx].quo_rem(diag[idx - 1])
        if r != 0:
            print(f'[ECHEC] {label}: d_{idx} ne divise pas d_{idx+1}')
            valide = False

    if valide:
        print(f'[OK]{label}')
    return valide



# Fonction qui test sur des cas simples

from sage.matrix.matrix_space import MatrixSpace

def tester_forme_smith():
    print('----Tests de forme_smith----')
    R = PolynomialRing(QQ, 'T')
    T = R.gen()

    cas = []

    A1 = matrix(R, [[T,0], [0,T-1]])
    cas.append((A1, 'Diagonale simple'))
    A2 = matrix(R, [[T,T], [T, T]])
    cas.append((A2, 'Rang 1'))
    A3 = matrix(R, [[T^2,T,1],[T,T^2,T],[1,T,T^2]])
    cas.append((A3, '3x3 symétrique'))


    M_jord = matrix(QQ, [[3,1,0],[0,3,1],[0,0,3]])
    EM = MatrixSpace(R, 3)
    mat_jord = EM.identity_matrix() * T - EM(M_jord)
    cas.append((mat_jord, 'Jordan J_3(3)'))
    A5 = matrix(R, [[T^2-1,T-1,0], [T+1,T^2-1,T-1]])
    cas.append((A5, 'Rectangulaire 2x3'))

    A6 = matrix(R, 3, 3)
    cas.append((A6, 'Matrice nulle'))

    def normalise_diag(D):
        R = D.base_ring()
        return [D[i,i] / D[i,i].leading_coefficient() if D[i,i] != 0 else R(0) for i in range(min(D.nrows(), D.ncols()))]









    tout_ok = True
    for A, label in cas:
        D, U, V = forme_smith(A)
        valide = _verifier_smith(A, D, U, V, label)
        tout_ok = tout_ok and valide

        D_ref, _, _ = A.smith_form()
        if normalise_diag(D) != normalise_diag(D_ref):
            print(f'  [DIFF]  {label} : diagonale différente de la référence SageMath')
            print(f'          perso : {normalise_diag(D)}')
            print(f'          natif : {normalise_diag(D_ref)}')
            tout_ok = False

    print()
    print('Résultat global :', 'TOUS OK' if tout_ok else 'ECHECS DÉTECTÉS')
    return tout_ok

In [90]:
tester_forme_smith()

----Tests de forme_smith----
[OK]Diagonale simple
[OK]Rang 1
[OK]3x3 symétrique
[OK]Jordan J_3(3)
[OK]Rectangulaire 2x3
[OK]Matrice nulle

Résultat global : TOUS OK


True

## 2. Invariants de similitude et rorme de Frobenius

In [ ]:
from sage.matrix.matrix_space import MatrixSpace

# Entrée: M \in M_n(k), corps k (défaut QQ)
# Sortie: dict avec
#'facteurs_invariants_smith': [d_1, ..., d_n] facteurs diagonaux unitaires de Smith(TI-M)
#'invariants_de_similitude': [P_1, ..., P_s] facteurs de degré >= 1, P_s | ... | P_1
#'factorisations': factorisations en irréductibles des P_i dans k[T]
#'smithD', 'U', 'V': forme de Smith et matrices de passage



def invariants_similitude(M, corps=QQ):
    n = M.nrows()
    assert M.ncols() == n

    R = PolynomialRing(corps, 'T')
    T = R.gen()
    EM = MatrixSpace(R, n)
    mat_caract = EM.identity_matrix() * T - EM(M)

    D, U, V = mat_caract.smith_form()

    facteurs = []
    for i in range(n):
        d = D[i, i]
        cd = d.leading_coefficient()
        facteurs.append(d / cd)

    invariants = [f for f in facteurs if f.degree() >= 1]
    invariants = list(reversed(invariants))

    return {'facteurs_invariants_smith': facteurs, 'invariants_de_similitude': invariants, 'factorisations': [inv.factor() for inv in invariants], 'smithD': D, 'U': U, 'V': V}





#Entrée: P \in k[T] unitaire de degré d
# Sortie: C(P) \in M_d(k), matrice compagnon dans la base (e_0, ..., e_{d-1}),
#           avec C(P) x e_i = e_{i+1} (si i < d-1) et sibib C(P)·e_{d-1} = -\sum_{i=0}^{d-1} P[i]·e_i.

def matrice_compagnon(P):
    d = P.degree()
    if d == 0:
        return matrix(P.base_ring(), 0, 0, [])
    k = P.parent().base_ring()
    C = matrix(k, d, d)
    for i in range(d - 1):
        C[i + 1, i] = 1
    for i in range(d):
        C[i, d - 1] = -P[i]
    return C




#Entrée: M \in M_n(k)
#Sortie: forme de Frobenius diag(C(P_s), ..., C(P_1)) \in M_n(k)

def frobenius(M, corps=QQ):
    res = invariants_similitude(M, corps)
    invs = res['invariants_de_similitude']
    blocs = [matrice_compagnon(P) for P in invs if P.degree() >= 1]
    if not blocs:
        return matrix(corps, 0, 0, [])
    return block_diagonal_matrix(blocs)

In [ ]:
#Affiche: M, forme de Smith de TI-M, facteurs invariants, invariants de similitude
#             avec factorisations, forme de Frobenius.
#Vérifie: produits(P_i) = \chi_M,  P_1 = \mu_M,  chaîne P_s | ... | P_1,  \chi_Frobenius = \chi_M.
#Retourne: le dict produit par invariants_similitude.



def afficher_resultats(M, corps=QQ, label=''):
    print()
    if label:
        print(label)
    print('Matrice M :')
    print(M)
    print()

    
    res = invariants_similitude(M, corps)
    invs = res['invariants_de_similitude']
    print('Forme de Smith de (T·Id - M) :')
    print(res['smithD'])
    print()

    print("Facteurs invariants de Smith (ordre croissant d'idéaux) :")
    for i, f in enumerate(res['facteurs_invariants_smith']):
        print(f'  d_{i+1} = {f}')
    print()

    print('Invariants de similitude  [P_s | ... | P_1,  P_1 = \mu_M] :')
    for i, inv in enumerate(invs):
        print(f'  P_{i+1} = {inv}')
    print()

    print('Factorisations :')
    for i, fac in enumerate(res['factorisations']):
        print(f'  P_{i+1} = {fac}')
    print()

    R = PolynomialRing(corps, 'T')
    chi = R(M.characteristic_polynomial())
    chi_norm = chi / chi.leading_coefficient()
    produit = product(invs) if invs else R(1)
    prod_norm = produit / produit.leading_coefficient()

    ok_chi = (R(prod_norm) == R(chi_norm))
    print(f'Vérification  produits des P_i = \chi_M : {ok_chi}')
    print(f'  produits des P_i = {prod_norm}')
    print(f'  \chi_M   = {chi_norm}')
    print()

    mu = M.minpoly()
    R2 = PolynomialRing(corps, 't')
    mu_norm = R2(mu) / mu.leading_coefficient()
    P1 = invs[0] if invs else None
    P1_R2 = R2(P1) if P1 is not None else None
    ok_min = (P1_R2 == mu_norm)
    print(f'Vérification  P_1 = \mu_M : {ok_min}')
    print(f'  P_1 = {P1}')
    print(f'  \mu_M = {mu_norm}')
    print()

    if len(invs) > 1:
        chaine_ok = all(invs[i] % invs[i-1] == 0 for i in range(1, len(invs)))
        print(f'Vérification chaîne  P_s | ... | P_1 : {chaine_ok}')
        print()

    F = frobenius(M, corps)
    print('Forme de Frobenius  diag(C(P_s), ..., C(P_1)) :')
    print(F)
    print()

    if F.nrows() > 0:
        R3 = PolynomialRing(corps, 'T')
        chi_F = R3(F.characteristic_polynomial())
        chi_F_norm = chi_F / chi_F.leading_coefficient()
        print(f'Vérification  \chi_Frobenius = \chi_M : {chi_F_norm == chi_norm}')

    return res

## Test sur des exemples

### Exemple 1 — $M = \text{ diagonale par blocs avec deux blocs } J_2(3)$


In [93]:
M1 = matrix(QQ, [
    [3, 1, 0, 0],
    [0, 3, 0, 0],
    [0, 0, 3, 1],
    [0, 0, 0, 3]
])
res1 = afficher_resultats(M1, QQ, '2 blocs J_2(3)')


2 blocs J_2(3)
Matrice M :
[3 1 0 0]
[0 3 0 0]
[0 0 3 1]
[0 0 0 3]

Forme de Smith de (T·Id - M) :
[            1             0             0             0]
[            0             1             0             0]
[            0             0 T^2 - 6*T + 9             0]
[            0             0             0 T^2 - 6*T + 9]

Facteurs invariants de Smith (ordre croissant d'idéaux) :
  d_1 = 1
  d_2 = 1
  d_3 = T^2 - 6*T + 9
  d_4 = T^2 - 6*T + 9

Invariants de similitude  [P_s | … | P_1,  P_1 = \mu_M] :
  P_1 = T^2 - 6*T + 9
  P_2 = T^2 - 6*T + 9

Factorisations :
  P_1 = (T - 3)^2
  P_2 = (T - 3)^2

Vérification  produits des P_i = \chi_M : True
  produits des P_i = T^4 - 12*T^3 + 54*T^2 - 108*T + 81
  \chi_M   = T^4 - 12*T^3 + 54*T^2 - 108*T + 81

Vérification  P_1 = \mu_M : True
  P_1 = T^2 - 6*T + 9
  \mu_M = t^2 - 6*t + 9

Vérification chaîne  P_s | … | P_1 : True

Forme de Frobenius  diag(C(P_s), …, C(P_1)) :
[ 0 -9| 0  0]
[ 1  6| 0  0]
[-----+-----]
[ 0  0| 0 -9]
[ 0  0| 1 

### Exemple 2 — $C(T^4 - 1)$


In [94]:
M2 = matrix(QQ, [
    [0, 0, 0, 1],
    [1, 0, 0, 0],
    [0, 1, 0, 0],
    [0, 0, 1, 0]
])
res2 = afficher_resultats(M2, QQ, 'Matrice compagnon de T^4 - 1')


Matrice compagnon de T^4 - 1
Matrice M :
[0 0 0 1]
[1 0 0 0]
[0 1 0 0]
[0 0 1 0]

Forme de Smith de (T·Id - M) :
[      1       0       0       0]
[      0       1       0       0]
[      0       0       1       0]
[      0       0       0 T^4 - 1]

Facteurs invariants de Smith (ordre croissant d'idéaux) :
  d_1 = 1
  d_2 = 1
  d_3 = 1
  d_4 = T^4 - 1

Invariants de similitude  [P_s | … | P_1,  P_1 = \mu_M] :
  P_1 = T^4 - 1

Factorisations :
  P_1 = (T - 1) * (T + 1) * (T^2 + 1)

Vérification  produits des P_i = \chi_M : True
  produits des P_i = T^4 - 1
  \chi_M   = T^4 - 1

Vérification  P_1 = \mu_M : True
  P_1 = T^4 - 1
  \mu_M = t^4 - 1

Forme de Frobenius  diag(C(P_s), …, C(P_1)) :
[0 0 0 1]
[1 0 0 0]
[0 1 0 0]
[0 0 1 0]

Vérification  \chi_Frobenius = \chi_M : True


### Matrice triangulaire $3 \times 3$

In [95]:
M3 = matrix(QQ, [
    [2, 1, 0],
    [0, 2, 0],
    [0, 0, 3]
])
res3 = afficher_resultats(M3, QQ, 'Exemple 3 — Matrice 3x3 triangulaire')


Exemple 3 — Matrice 3x3 triangulaire
Matrice M :
[2 1 0]
[0 2 0]
[0 0 3]

Forme de Smith de (T·Id - M) :
[                      1                       0                       0]
[                      0                       1                       0]
[                      0                       0 T^3 - 7*T^2 + 16*T - 12]

Facteurs invariants de Smith (ordre croissant d'idéaux) :
  d_1 = 1
  d_2 = 1
  d_3 = T^3 - 7*T^2 + 16*T - 12

Invariants de similitude  [P_s | … | P_1,  P_1 = \mu_M] :
  P_1 = T^3 - 7*T^2 + 16*T - 12

Factorisations :
  P_1 = (T - 3) * (T - 2)^2

Vérification  produits des P_i = \chi_M : True
  produits des P_i = T^3 - 7*T^2 + 16*T - 12
  \chi_M   = T^3 - 7*T^2 + 16*T - 12

Vérification  P_1 = \mu_M : True
  P_1 = T^3 - 7*T^2 + 16*T - 12
  \mu_M = t^3 - 7*t^2 + 16*t - 12

Forme de Frobenius  diag(C(P_s), …, C(P_1)) :
[  0   0  12]
[  1   0 -16]
[  0   1   7]

Vérification  \chi_Frobenius = \chi_M : True


### Double rotation de $\pi/2$

In [96]:
M4 = matrix(QQ, [
    [0, -1, 0,  0],
    [1,  0, 0,  0],
    [0,  0, 0, -1],
    [0,  0, 1,  0]
])
res4 = afficher_resultats(M4, QQ, 'Exemple 4 — Double rotation de pi/2')


Exemple 4 — Double rotation de pi/2
Matrice M :
[ 0 -1  0  0]
[ 1  0  0  0]
[ 0  0  0 -1]
[ 0  0  1  0]

Forme de Smith de (T·Id - M) :
[      1       0       0       0]
[      0       1       0       0]
[      0       0 T^2 + 1       0]
[      0       0       0 T^2 + 1]

Facteurs invariants de Smith (ordre croissant d'idéaux) :
  d_1 = 1
  d_2 = 1
  d_3 = T^2 + 1
  d_4 = T^2 + 1

Invariants de similitude  [P_s | … | P_1,  P_1 = \mu_M] :
  P_1 = T^2 + 1
  P_2 = T^2 + 1

Factorisations :
  P_1 = T^2 + 1
  P_2 = T^2 + 1

Vérification  produits des P_i = \chi_M : True
  produits des P_i = T^4 + 2*T^2 + 1
  \chi_M   = T^4 + 2*T^2 + 1

Vérification  P_1 = \mu_M : True
  P_1 = T^2 + 1
  \mu_M = t^2 + 1

Vérification chaîne  P_s | … | P_1 : True

Forme de Frobenius  diag(C(P_s), …, C(P_1)) :
[ 0 -1| 0  0]
[ 1  0| 0  0]
[-----+-----]
[ 0  0| 0 -1]
[ 0  0| 1  0]

Vérification  \chi_Frobenius = \chi_M : True


### Matrice nilpotente

In [97]:
M5 = matrix(QQ, [
    [0, 0, 0],
    [1, 0, 0],
    [0, 1, 0]
])
res5 = afficher_resultats(M5, QQ, 'Matrice nilpotente ')


Matrice nilpotente 
Matrice M :
[0 0 0]
[1 0 0]
[0 1 0]

Forme de Smith de (T·Id - M) :
[  1   0   0]
[  0   1   0]
[  0   0 T^3]

Facteurs invariants de Smith (ordre croissant d'idéaux) :
  d_1 = 1
  d_2 = 1
  d_3 = T^3

Invariants de similitude  [P_s | … | P_1,  P_1 = \mu_M] :
  P_1 = T^3

Factorisations :
  P_1 = T^3

Vérification  produits des P_i = \chi_M : True
  produits des P_i = T^3
  \chi_M   = T^3

Vérification  P_1 = \mu_M : True
  P_1 = T^3
  \mu_M = t^3

Forme de Frobenius  diag(C(P_s), …, C(P_1)) :
[0 0 0]
[1 0 0]
[0 1 0]

Vérification  \chi_Frobenius = \chi_M : True
